# Проверка DataLoader

- Индекс и split
- Формы batch
- NaN и Inf


In [ ]:
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader

from mden_battery.data import (
    PreparedWindowIterableDataset,
    WindowConfig,
)


In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data" / "prepared_log_age"
INDEX_PATH = DATA_ROOT / "prepared_index.csv"
SCALER_PATH = DATA_ROOT / "scaler.csv"

INPUT_COLUMNS = [
    "cycle",
    "time_s",
    "voltage_V",
    "current_A",
    "temperature_C",
]
WINDOW_CONFIG = WindowConfig(
    input_length=32,
    horizon=8,
    stride=8,
)
BATCH_SIZE = 32
NUM_WORKERS = 0
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


In [ ]:
def load_index(path: Path) -> pd.DataFrame:
    """Load and validate the prepared-part index."""
    if not path.is_file():
        raise FileNotFoundError(path)
    index = pd.read_csv(path)
    required = {
        "cell_id",
        "split",
        "source",
        "path",
        "rows",
        "chunk",
    }
    missing = required - set(index.columns)
    if missing:
        raise KeyError(f"Index misses columns: {sorted(missing)}")
    if index.empty:
        raise ValueError("Prepared index is empty")
    return index


In [ ]:
def build_loader(split: str) -> DataLoader:
    """Build a streaming loader for one dataset split."""
    dataset = PreparedWindowIterableDataset(
        INDEX_PATH,
        split=split,
        input_cols=INPUT_COLUMNS,
        config=WINDOW_CONFIG,
        scaler_csv=SCALER_PATH,
    )
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
    )


## Индекс


In [ ]:
assert (PROJECT_ROOT / "pyproject.toml").is_file()
assert SCALER_PATH.is_file(), SCALER_PATH

index = load_index(INDEX_PATH)
missing_paths = [
    path for path in index["path"] if not Path(path).is_file()
]
assert not missing_paths, missing_paths[:5]
assert "train" in set(index["split"])

split_summary = (
    index.groupby("split", as_index=False)
    .agg(parts=("path", "size"), rows=("rows", "sum"))
    .sort_values("split")
)
print(split_summary.to_string(index=False))


## Batch


In [ ]:
train_loader = build_loader("train")
batch = next(iter(train_loader))

assert batch["x"].ndim == 3
assert batch["x"].shape[1:] == (32, 5)
assert batch["y_soc"].shape[1:] == (8,)
assert batch["y_soh"].shape[1:] == (8,)
assert all(torch.isfinite(tensor).all() for tensor in batch.values())

print("device:", DEVICE)
print("x:", tuple(batch["x"].shape))
print("y_soc:", tuple(batch["y_soc"].shape))
print("y_soh:", tuple(batch["y_soh"].shape))
